# 🐾 SmartZoo 3: ระบบเตรียมชุดข้อมูลภาพ AI สัตว์เลี้ยงลูกด้วยนมในสวนสัตว์ไทย (250 สายพันธุ์)
**Thai Zoo Mammals Dataset & Image Harvester for AI Model Training**

สมุดบันทึกนี้ถูกพัฒนาขึ้นใหม่เพื่อต่อยอดจากฐานข้อมูล **`mammals.json`** (250 สายพันธุ์ที่คัดกรองและเรียงตามชื่อไทย ก-ฮ แล้ว) โดยไม่ต้องรันค้นหาจากสัตว์นับหมื่นชนิดทั่วโลกเหมือนเวอร์ชันเดิม:
1. **จับคู่ชื่อวิทยาศาสตร์ (Taxon Matching):** ค้นหา `taxonKey` สากลจากชื่อวิทยาศาสตร์ในฐานข้อมูลชีววิทยาโลก GBIF API
2. **สำรวจจำนวนภาพ (Image Availability):** ตรวจสอบจำนวนภาพถ่ายจริง (`StillImage`) ที่พร้อมนำมาใช้เทรนโมเดล
3. **ดาวน์โหลดภาพสำหรับเทรน (Image Harvester):** ดาวน์โหลดภาพจริงลงโฟลเดอร์ตามโครงสร้างมาตรฐาน Image Classification (พร้อมใช้กับ TensorFlow / PyTorch / YOLO)
4. **ตรวจสอบคุณภาพภาพ (Visual Inspection):** สุ่มแสดงภาพที่ดาวน์โหลดมาตรวจสอบความถูกต้อง


### 1. ติดตั้งและนำเข้าไลบรารีที่จำเป็น (Setup & Import Libraries)
รองรับการรันได้ทั้งบน **Google Colab** และ **เครื่องส่วนตัว (Local VS Code / JupyterLab)**


In [ ]:
import os
import sys
import json
import time
import requests
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from tqdm.notebook import tqdm
from PIL import Image
from io import BytesIO

# ตรวจสอบสภาพแวดล้อม (Google Colab vs Local)
IS_COLAB = 'google.colab' in sys.modules

if IS_COLAB:
    from google.colab import drive
    print("💻 กำลังรันบน Google Colab -> กำลังเชื่อมต่อ Google Drive...")
    drive.mount('/content/drive')
    BASE_DIR = "/content/drive/MyDrive/Colab Notebooks/SmartZoo"
else:
    print("💻 กำลังรันบน Local Environment")
    BASE_DIR = "."

print("✅ โหลดไลบรารีเรียบร้อยแล้ว")


### 2. โหลดข้อมูลสัตว์ 250 สายพันธุ์จาก `mammals.json`
ระบบจะค้นหาไฟล์ `mammals.json` โดยอัตโนมัติ


In [ ]:
# ค้นหาตำแหน่งไฟล์ mammals.json
candidate_paths = [
    os.path.join(BASE_DIR, "data", "mammals.json"),
    os.path.join(BASE_DIR, "data", "species", "mammals.json"),
    "data/mammals.json",
    "data/species/mammals.json",
    "../data/mammals.json"
]

json_path = None
for p in candidate_paths:
    if os.path.exists(p):
        json_path = p
        break

if not json_path:
    raise FileNotFoundError("❌ ไม่พบไฟล์ mammals.json กรุณาตรวจสอบว่ามีไฟล์อยู่ในโฟลเดอร์ data/")

with open(json_path, "r", encoding="utf-8") as f:
    mammals_data = json.load(f)

print(f"✅ โหลดข้อมูลสำเร็จ: พบสัตว์เลี้ยงลูกด้วยนมในสวนสัตว์ไทยทั้งหมด {len(mammals_data)} สายพันธุ์")

# แปลงเป็น DataFrame เพื่อดูตัวอย่างข้อมูล
df_mammals = pd.DataFrame(mammals_data)
display(df_mammals[["id", "thai_name", "english_name", "scientific_name", "family", "iucn_status"]].head(8))


### 3. ตรวจสอบ TaxonKey และจำนวนรูปภาพใน GBIF API
ยิง API ไปยัง GBIF เพื่อหา:
1. `taxonKey` สากลจากชื่อวิทยาศาสตร์ (`scientific_name`)
2. จำนวนรูปภาพ (`gbifImageCount`) ในฐานข้อมูล GBIF
*(มีระบบ Checkpoint บันทึกสำรองอัตโนมัติลงไฟล์ `mammals_with_gbif_images.json`)*


In [ ]:
output_path = os.path.join(os.path.dirname(json_path), "mammals_with_gbif_images.json")

# ตรวจสอบข้อมูลที่เคยดึงไว้แล้ว (Resume Checkpoint)
results_dict = {}
if os.path.exists(output_path):
    try:
        with open(output_path, "r", encoding="utf-8") as f:
            prev_results = json.load(f)
            results_dict = {item["id"]: item for item in prev_results}
        print(f"📊 พบข้อมูลเดิมที่เคยดึงไว้แล้ว {len(results_dict)} รายการ")
    except Exception as e:
        print(f"⚠️ โหลดข้อมูลเดิมไม่สำเร็จ ({e}) จะเริ่มดึงใหม่")

def match_gbif_species(scientific_name):
    """ค้นหา taxonKey จากชื่อวิทยาศาสตร์ใน GBIF Backbone Taxonomy"""
    url = f"https://api.gbif.org/v1/species/match"
    params = {"name": scientific_name, "strict": "false"}
    headers = {"User-Agent": "SmartZooResearchBot/2.0 (contact@smartzoo.local)"}
    try:
        res = requests.get(url, params=params, headers=headers, timeout=10)
        if res.status_code == 200:
            data = res.json()
            return data.get("usageKey") or data.get("speciesKey"), data.get("matchType")
    except Exception:
        pass
    return None, "NONE"

def get_image_count(taxon_key):
    """สอบถามจำนวนรูปภาพที่มีในฐานข้อมูล GBIF"""
    if not taxon_key:
        return 0
    url = "https://api.gbif.org/v1/occurrence/search"
    params = {"taxonKey": taxon_key, "mediaType": "StillImage", "limit": 0}
    headers = {"User-Agent": "SmartZooResearchBot/2.0 (contact@smartzoo.local)"}
    try:
        res = requests.get(url, params=params, headers=headers, timeout=10)
        if res.status_code == 200:
            return res.json().get("count", 0)
    except Exception:
        pass
    return 0

print("🚀 กำลังตรวจสอบข้อมูลกับ GBIF API...")
for animal in tqdm(mammals_data):
    a_id = animal["id"]
    
    # หากเคยดึงข้อมูลไว้แล้ว ให้ข้าม
    if a_id in results_dict and "gbifImageCount" in results_dict[a_id]:
        animal["taxonKey"] = results_dict[a_id].get("taxonKey")
        animal["matchType"] = results_dict[a_id].get("matchType")
        animal["gbifImageCount"] = results_dict[a_id].get("gbifImageCount", 0)
        continue
        
    sci_name = animal.get("scientific_name", "").strip()
    taxon_key, match_type = match_gbif_species(sci_name)
    img_count = get_image_count(taxon_key)
    
    animal["taxonKey"] = taxon_key
    animal["matchType"] = match_type
    animal["gbifImageCount"] = img_count
    results_dict[a_id] = animal
    
    # บันทึกสำรองทุก 25 รายการ
    if len(results_dict) % 25 == 0:
        with open(output_path, "w", encoding="utf-8") as f:
            json.dump(list(results_dict.values()), f, ensure_ascii=False, indent=2)
            
    time.sleep(0.05)

# บันทึกไฟล์สมบูรณ์
with open(output_path, "w", encoding="utf-8") as f:
    json.dump(mammals_data, f, ensure_ascii=False, indent=2)

print(f"\n🎉 สำเร็จ! ตรวจสอบครบ {len(mammals_data)} ชนิด บันทึกที่: {output_path}")


### 4. วิเคราะห์และสรุปผลความพร้อมของข้อมูลภาพ (Analysis & Visualization)


In [ ]:
df_analyzed = pd.DataFrame(mammals_data)
df_sorted_img = df_analyzed.sort_values(by="gbifImageCount", ascending=False).reset_index(drop=True)

print("🏆 10 อันดับสัตว์ในสวนสัตว์ไทยที่มีรูปภาพมากที่สุดใน GBIF:")
display(df_sorted_img[["id", "thai_name", "english_name", "scientific_name", "gbifImageCount"]].head(10))

# สถิติความพร้อมของรูปภาพ
total = len(df_analyzed)
c_100 = len(df_analyzed[df_analyzed["gbifImageCount"] >= 100])
c_50 = len(df_analyzed[df_analyzed["gbifImageCount"] >= 50])
c_10 = len(df_analyzed[df_analyzed["gbifImageCount"] >= 10])

print(f"\n📊 สถิติความพร้อมของรูปภาพสำหรับเทรน AI:")
print(f"- สายพันธุ์ที่มีภาพ >= 100 ภาพ (พร้อมเทรน CNN คุณภาพสูง): {c_100}/{total} ชนิด ({c_100/total*100:.1f}%)")
print(f"- สายพันธุ์ที่มีภาพ >= 50 ภาพ (พร้อมเทรน Transfer Learning / MobileNet): {c_50}/{total} ชนิด ({c_50/total*100:.1f}%)")
print(f"- สายพันธุ์ที่มีภาพ >= 10 ภาพ: {c_10}/{total} ชนิด ({c_10/total*100:.1f}%)")

# พล็อต Bar Chart แสดง 15 อันดับแรก
plt.figure(figsize=(12, 6))
top15 = df_sorted_img.head(15)
plt.barh(top15["english_name"] + " (" + top15["thai_name"] + ")", top15["gbifImageCount"], color="#2b8a3e")
plt.gca().invert_yaxis()
plt.title("Top 15 Mammals in Thai Zoos by GBIF Image Count", fontsize=14, pad=15)
plt.xlabel("Number of Images in GBIF")
plt.grid(axis="x", linestyle="--", alpha=0.7)
plt.tight_layout()
plt.show()


### 5. ระบบดาวน์โหลดภาพสำหรับเทรนโมเดล AI (Image Harvester)
ฟังก์ชันดาวน์โหลดรูปภาพจริงจาก GBIF โดยจัดโครงสร้างโฟลเดอร์ตามคลาสมาตรฐาน:
```
dataset/
  mammals_training/
    001_Tragulus_napu/
    008_Bos_gaurus/
    ...
```
*(สามารถปรับตัวแปร `TARGET_SPECIES_COUNT` เพื่อทดสอบดาวน์โหลดทีละน้อยก่อนได้)*


In [ ]:
DATASET_DIR = os.path.join(BASE_DIR, "dataset", "mammals_training")
os.makedirs(DATASET_DIR, exist_ok=True)

# กำหนดเป้าหมายการดาวน์โหลด:
TARGET_SPECIES_COUNT = 5     # จำนวนสายพันธุ์ที่ต้องการทดสอบดาวน์โหลด (กำหนดเป็น None หากต้องการทุกสายพันธุ์)
MAX_IMAGES_PER_SPECIES = 20  # จำนวนรูปภาพสูงสุดต่อสายพันธุ์

def download_species_images(animal_info, max_images=20):
    """ดาวน์โหลดรูปภาพของสปีชีส์ที่ระบุจาก GBIF Occurrence API"""
    taxon_key = animal_info.get("taxonKey")
    if not taxon_key or animal_info.get("gbifImageCount", 0) == 0:
        return 0
        
    sci_name_clean = animal_info["scientific_name"].replace(" ", "_").replace("/", "_")
    class_folder = f"{animal_info['id']:03d}_{sci_name_clean}"
    save_dir = os.path.join(DATASET_DIR, class_folder)
    os.makedirs(save_dir, exist_ok=True)
    
    url = "https://api.gbif.org/v1/occurrence/search"
    params = {"taxonKey": taxon_key, "mediaType": "StillImage", "limit": max_images * 2}
    headers = {"User-Agent": "SmartZooResearchBot/2.0 (contact@smartzoo.local)"}
    
    downloaded_count = 0
    try:
        res = requests.get(url, params=params, headers=headers, timeout=15)
        if res.status_code != 200:
            return 0
        data = res.json()
        results = data.get("results", [])
        
        for item in results:
            if downloaded_count >= max_images:
                break
            media_list = item.get("media", [])
            for media in media_list:
                img_url = media.get("identifier")
                if not img_url:
                    continue
                try:
                    img_res = requests.get(img_url, headers=headers, timeout=10)
                    if img_res.status_code == 200:
                        img = Image.open(BytesIO(img_res.content)).convert("RGB")
                        file_name = f"img_{downloaded_count+1:03d}.jpg"
                        img.save(os.path.join(save_dir, file_name), "JPEG", quality=90)
                        downloaded_count += 1
                        break
                except Exception:
                    continue
    except Exception as e:
        print(f"⚠️ ข้อผิดพลาดในการดาวน์โหลด {animal_info['thai_name']}: {e}")
        
    return downloaded_count

# คัดเลือกสายพันธุ์ที่มีรูปภาพมาทดสอบดาวน์โหลด
if TARGET_SPECIES_COUNT:
    target_list = df_sorted_img.head(TARGET_SPECIES_COUNT).to_dict("records")
else:
    target_list = df_sorted_img.to_dict("records")

print(f"📥 เริ่มต้นดาวน์โหลดภาพสำหรับ {len(target_list)} สายพันธุ์ (สูงสุด {MAX_IMAGES_PER_SPECIES} ภาพ/สายพันธุ์)...")
for animal in tqdm(target_list):
    cnt = download_species_images(animal, max_images=MAX_IMAGES_PER_SPECIES)
    print(f"  ✅ {animal['thai_name']} ({animal['scientific_name']}): ดาวน์โหลดได้ {cnt} ภาพ")

print(f"\n🎉 ดาวน์โหลดชุดภาพเรียบร้อยแล้ว บันทึกไว้ที่: {DATASET_DIR}")


### 6. แสดงตัวอย่างภาพที่ดาวน์โหลดมา (Visual Inspection)
สุ่มภาพจากโฟลเดอร์ Dataset มาแสดงผล เพื่อตรวจสอบคุณภาพก่อนนำไปเข้าโมเดล


In [ ]:
import glob
import random

all_downloaded_images = glob.glob(os.path.join(DATASET_DIR, "*", "*.jpg"))
print(f"🖼️ รวมภาพที่ดาวน์โหลดมาทั้งหมดในคลัง: {len(all_downloaded_images)} ภาพ")

if len(all_downloaded_images) > 0:
    sample_size = min(8, len(all_downloaded_images))
    sample_images = random.sample(all_downloaded_images, sample_size)
    
    fig, axes = plt.subplots(2, 4, figsize=(16, 8))
    axes = axes.flatten()
    
    for idx, img_path in enumerate(sample_images):
        img = Image.open(img_path)
        class_name = os.path.basename(os.path.dirname(img_path))
        axes[idx].imshow(img)
        axes[idx].set_title(class_name, fontsize=9)
        axes[idx].axis("off")
        
    plt.tight_layout()
    plt.show()
else:
    print("ยังไม่มีภาพที่ดาวน์โหลด กรุณารันขั้นตอนที่ 5 ก่อน")
